# Task 5 — Step 1: Entity Sampling for Web Retrieval

Samples entities from both datasets (stratified by `brand` for WDC Products, `venue` for
DBLP-Scholar) to ensure source/category diversity, and builds a web search query string for
each sampled entity. Output feeds into Step 2 (Tavily search).

In [1]:
PER_SIDE = 150              # entities sampled per side (left/right) -> 300 total per dataset
BUCKET_CAP_FRACTION = 0.15  # no single brand/venue contributes more than this fraction of a side's sample
RANDOM_SEED = 42

In [2]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
rng = np.random.default_rng(RANDOM_SEED)

In [3]:
def stratified_sample(df: pd.DataFrame, bucket_col: str, n: int, cap_fraction: float, rng: np.random.Generator) -> pd.DataFrame:
    """Sample n rows from df, capping how many rows any single bucket value can contribute.

    First draws up to `cap` rows per bucket, then tops up from the remaining pool
    (uniformly at random) until n rows are reached or the pool is exhausted.
    """
    n = min(n, len(df))
    cap = max(1, int(np.ceil(n * cap_fraction)))

    picked_idx = []
    for _, group in df.groupby(bucket_col, dropna=False):
        take = min(cap, len(group))
        picked_idx.extend(rng.choice(group.index.to_numpy(), size=take, replace=False))

    picked_idx = list(dict.fromkeys(picked_idx))  # de-dupe, preserve order

    if len(picked_idx) > n:
        picked_idx = list(rng.choice(picked_idx, size=n, replace=False))
    elif len(picked_idx) < n:
        remaining = df.index.difference(picked_idx)
        top_up = rng.choice(remaining.to_numpy(), size=n - len(picked_idx), replace=False)
        picked_idx.extend(top_up)

    return df.loc[picked_idx].copy()

In [4]:
def strip_col_val(text: str) -> str:
    return re.sub(r"COL \S+ VAL\s*", " ", str(text)).strip()


def _clean(val) -> str:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return ""
    return str(val).strip()


def build_wdc_query(row: pd.Series) -> str:
    brand = _clean(row.get("brand"))
    title = _clean(row.get("title"))
    return f"{brand} {title}".strip()


def build_dblp_query(row: pd.Series) -> str:
    title = _clean(row.get("title"))
    authors = _clean(row.get("authors"))
    venue = _clean(row.get("venue"))
    first_author = authors.split(",")[0].strip() if authors else ""
    return " ".join(p for p in [title, first_author, venue] if p)

In [ ]:
# Entity-disjoint requirement:
# web queries must be built only from TRAIN entities, never valid/test, so the
# retrieved + labeled pairs cannot leak held-out entities into the training set.
from src.data_prep.entity_splits import train_entity_ids

DATASET_CONFIG = {
    "wdc-products": {"bucket_col": "brand", "query_fn": build_wdc_query},
    "dblp-scholar": {"bucket_col": "venue", "query_fn": build_dblp_query},
}

summary = {}

for dataset, cfg in DATASET_CONFIG.items():
    print(f"\n{'='*60}\n  {dataset}\n{'='*60}")
    out_dir = PROCESSED / dataset
    bucket_col = cfg["bucket_col"]
    query_fn = cfg["query_fn"]

    train_left_ids, train_right_ids = train_entity_ids(dataset)

    side_samples = []
    for side in ["left", "right"]:
        df = pd.read_csv(out_dir / f"entities_{side}.csv")

        # Restrict to train-only entities before sampling (entity-disjoint).
        allowed = train_left_ids if side == "left" else train_right_ids
        n_all = len(df)
        df = df[df["id"].astype(int).isin(allowed)].reset_index(drop=True)
        print(f"[{side}] train-only entities: {len(df):,} / {n_all:,}")

        # Compute the query from the original (un-modified) attribute values first,
        # then create a separate column for stratification bucketing.
        df["query"] = df.apply(query_fn, axis=1)
        df["_bucket"] = df[bucket_col].fillna("_none").astype(str).str.strip().replace("", "_none")

        sample = stratified_sample(df, "_bucket", PER_SIDE, BUCKET_CAP_FRACTION, rng)
        sample["side"] = side
        side_samples.append(sample)

        n_buckets = sample["_bucket"].nunique()
        top_buckets = sample["_bucket"].value_counts().head(5)
        print(f"\n[{side}] sampled {len(sample)} entities across {n_buckets} distinct '{bucket_col}' values")
        print(f"  top values:\n{top_buckets.to_string()}")

    combined = pd.concat(side_samples, ignore_index=True)
    combined = combined[["id", "side", "_bucket", "text", "query"]].rename(columns={"_bucket": "bucket"})

    empty_queries = (combined["query"].str.strip() == "").sum()
    if empty_queries:
        print(f"\n[WARNING] {empty_queries} entities produced an empty query string")

    out_path = out_dir / "web_query_entities.csv"
    combined.to_csv(out_path, index=False)
    print(f"\nSaved {len(combined)} sampled entities -> {out_path}")
    summary[dataset] = combined


  wdc-products
[left] train-only entities: 947 / 2,896

[left] sampled 150 entities across 78 distinct 'brand' values
  top values:
_bucket
_none            8
Epson            8
Corsair          8
Maxxis           5
Cooler Master    5
[right] train-only entities: 940 / 2,891

[right] sampled 150 entities across 75 distinct 'brand' values
  top values:
_bucket
_none            12
Epson             9
Corsair           8
Samsung           5
Cooler Master     5

Saved 300 sampled entities -> /Users/abd/Developer/thesis-project/data/processed/wdc-products/web_query_entities.csv

  dblp-scholar
[left] train-only entities: 2,382 / 2,616

[left] sampled 150 entities across 7 distinct 'venue' values
  top values:
_bucket
sigmod conference              28
sigmod record                  27
vldb                           25
_none                          23
acm trans . database syst .    23
[right] train-only entities: 8,058 / 64,263

[right] sampled 150 entities across 136 distinct 'venue' value

In [6]:
# Sanity check: preview a few sampled queries per dataset
for dataset, df in summary.items():
    print(f"\n{'='*60}\n  {dataset} — sample queries\n{'='*60}")
    for _, row in df.sample(5, random_state=RANDOM_SEED).iterrows():
        print(f"  [{row.side:5s}] bucket={row.bucket!r:25s} query={row.query!r}")


  wdc-products — sample queries
  [right] bucket='_none'                   query='INTEL CPU 10TH GEN COMET LAKE I7-10700KF 3.80GHZ LGA1200 16.00MB CACHE BOXED'
  [right] bucket='Jabra'                   query='Jabra Jabra EVOLVE 65 UC Mono Bluetooth Headset'
  [right] bucket='Samsung'                 query='Samsung Samsung 860 PRO 1TB SSD 2-bit MLC V-NAND SATA III 6Gb/s 2.5 Internal Solid State Drive'
  [left ] bucket='Cisco'                   query='Cisco Cisco Catalyst 2960-X | WS-C2960X-24TD-L Catalyst 2960X 24 port 10/100/1000, 2 x 10G SFP+, LAN Base'
  [right] bucket='SanDisk'                 query='SanDisk SanDisk Ultra SDXC Memory Card 100MBs Class 10 UHS-I 256GB'

  dblp-scholar — sample queries
  [right] bucket='vldb ,'                  query='grid and applications ( industrial session ) f leymann vldb ,'
  [right] bucket='submitted for publication ,' query='scalable query processing in dynamic and open environments : an approach based on information e mena submitted for publ